In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv("/kaggle/input/q1-ka-ai-2026/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.hist(df["Delivery_Time"],bins=50)
plt.title("Delivery_Time Distrubution")
plt.xlabel("Delivery_Time")
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data
df = df.drop("Order_ID",axis=1)

In [ ]:
# Task 2: Write your code here: Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
df_missing = df.dropna(subset=["Delivery_Time"]).copy()
for cate in ["Weather","Traffic_Level","Time_of_Day"]:
  df_missing[cate] = df_missing[cate].fillna("unknown")

df_missing["Courier_Experience_yrs"] = df_missing["Courier_Experience_yrs"].fillna(df_missing["Courier_Experience_yrs"].mode()[0])
df_missing.info()




In [ ]:
# Task 3: Write your code here: Check and remove duplicates if any exist
df_clean = df_missing.copy()
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import OneHotEncoder

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
print(list(categorical_cols))


# df_encoded = pd.DataFrame(onehot_encoder.fit_transform(df_clean[categorical_cols]), columns=onehot_encoder.get_feature_names_out(df_clean[categorical_cols].columns))

# df_clean_encoded = pd.concat([df_clean.drop(categorical_cols,axis=1),df_encoded]).copy()
# df_clean_encoded
df_encoded = pd.get_dummies(df_clean,columns=categorical_cols,drop_first=True,dtype="int64")
df_encoded


In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

features = df_encoded.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_encoded[features] = scaler.fit_transform(df_encoded[features])
df_encoded.head()

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)


In [ ]:
# Task 1: Write your code here:
X = df_encoded.drop("Delivery_Time", axis=1).astype(float)
y = df_encoded['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
import numpy as np

n_splits = 5
rf = RandomForestRegressor(n_estimators=200)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_mae = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  print(f"Training Random Forest...")

    # Train
  rf.fit(X_train, y_train)

    # Predict
  y_pred = rf.predict(X_test)

    # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

  all_mae.append(mae)


print(f"  MAE:  {np.mean(all_mae):.4f}")



In [ ]:
# Task 1: Write your code here:
fi = pd.DataFrame({"Random Forest":rf.feature_importances_},index=rf.feature_names_in_)
imp = fi["Random Forest"]
sorted_idx = np.argsort(imp)
plt.barh(features[sorted_idx], imp[sorted_idx])

plt.title(f"RF Feature Importance")
plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(6,4))
plt.hist(y_pred)
plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:
%pip install catboost
from catboost import CatBoostRegressor
models = {  "Random Forest": RandomForestRegressor(n_estimators=320),
          "CatBoost": CatBoostRegressor(verbose=0)
            }

n_splits = 5
rf = RandomForestRegressor(n_estimators=200)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_mae_rf = []
all_mae_cat = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  for model_name,model in models.items():
    print(f"Training {model_name}...")


    # Train
    model.fit(X_train, y_train)

      # Predict
    y_pred = model.predict(X_test)

      # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    all_mae.append(mae)
